<a href="https://colab.research.google.com/github/AI-Object-Dedection/sam3/blob/main/colab_faz4_egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/AI-Object-Dedection/sam3/blob/main/colab_faz4_egitim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SAM3 Faz 4 - Tam Egitim (Colab Pro+ / Arka Plan)

Bu notebook, Kaggle'da yasanan iki sorunu cozer **ve** Colab'da yasadigimiz
Drive baglanti kopmasi (`Transport endpoint is not connected`) sorununu onler.

**Mimari (neden artik cokmuyor):**
- **Kod ve veri seti yerel diskte** (`/content`) tutulur -> egitim Drive'dan hic okumaz,
  bu yuzden Drive mount'u cokmez ve `%cd` hatasi olmaz.
- **Checkpoint ve loglar yerel diske yazilir**, ayri bir **yedekleme dongusu**
  her 120 saniyede bir bunlari **Drive'a kopyalar**. Yani egitim sureci Drive'a
  hic dokunmaz -> Drive kopsa bile **egitim DURMAZ**, ilerleme yine de Drive'da birikir.
- Egitim **arka plan process** olarak calisir -> sekme/baglanti kapansa bile devam eder.

**ONEMLI:** Calistirmadan once `Runtime > Change runtime type > A100/L4 GPU` sec.

## Adim 0 - GPU Kontrolu

In [3]:
import torch

if torch.cuda.is_available():
    print(f"GPU aktif: {torch.cuda.get_device_name(0)}")
    print(f"GPU bellek: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("UYARI: GPU bulunamadi! Runtime > Change runtime type > GPU sec.")

GPU aktif: NVIDIA A100-SXM4-40GB
GPU bellek: 42.4 GB


## Adim 1 - Google Drive Baglantisi ve Yollar

Kalici olmasi gerekenler (veri seti, checkpoint, log) Drive'da `MyDrive/sam3/` altinda.
Egitimin hizli ve guvenli calismasi icin kod + veri **yerel diske** (`/content`) alinir.

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

DRIVE_ROOT = "/content/drive/MyDrive/sam3"

# --- KALICI (Drive) - kaybolmamasi gereken her sey burada ---
DRIVE_DATA_DIR       = f"{DRIVE_ROOT}/dacl10k/"       # veri setinin kalici kopyasi
DRIVE_CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints/"   # checkpointler buraya yedeklenir
DRIVE_LOG_DIR        = f"{DRIVE_ROOT}/logs/"          # loglar buraya yedeklenir

# --- YEREL DISK (hizli, FUSE kopmaz) - egitim bunlari kullanir ---
REPO_DIR       = "/content/SAM3"          # kod yerel diskte (cwd hatasi olmaz)
LOCAL_DATA_DIR = "/content/dacl10k/"      # egitim veriyi buradan okur
LOCAL_CKPT_DIR = "/content/checkpoints/"  # egitim checkpointleri buraya yazar
LOCAL_LOG_DIR  = "/content/logs/"         # egitim logu buraya yazilir

for d in (DRIVE_CHECKPOINT_DIR, DRIVE_LOG_DIR, LOCAL_CKPT_DIR, LOCAL_LOG_DIR):
    os.makedirs(d, exist_ok=True)

print("Drive baglandi!")
print("Kalici veri (Drive)   :", DRIVE_DATA_DIR)
print("Kalici checkpoint     :", DRIVE_CHECKPOINT_DIR)
print("Yerel repo (egitim)   :", REPO_DIR)
print("Yerel veri (egitim)   :", LOCAL_DATA_DIR)

Mounted at /content/drive
Drive baglandi!
Kalici veri (Drive)   : /content/drive/MyDrive/sam3/dacl10k/
Kalici checkpoint     : /content/drive/MyDrive/sam3/checkpoints/
Yerel repo (egitim)   : /content/SAM3
Yerel veri (egitim)   : /content/dacl10k/


## Adim 2 - Proje Kodunu Hazirla (Yerel Diske)

Repo **yerel diske** (`/content/SAM3`) klonlanir - Drive'a bagimli olmadigi icin
`Transport endpoint is not connected` / `%cd` hatasi olusmaz. Kod zaten GitHub'da
kalici oldugu icin her oturumda yeniden klonlamak sorun degil.

`torchao` paketi `peft` ile uyumsuz (`ImportError`), bu yuzden kaldirilir;
`peft` torchao yoksa o kontrolu atlar ve LoRA normal sekilde uygulanir.

In [6]:
REPO_URL = "https://github.com/AI-Object-Dedection/sam3.git"

if not os.path.isdir(REPO_DIR):
    print("Repo yerel diske klonlaniyor...")
    !git clone {REPO_URL} "{REPO_DIR}"
else:
    print("Repo yerel diskte mevcut, guncelleniyor...")
    !cd "{REPO_DIR}" && git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt
# peft ile uyumsuz torchao'yu kaldir (LoRA icin gerekli degil)
!pip uninstall -y -q torchao

Repo yerel diske klonlaniyor...
Cloning into '/content/SAM3'...
remote: Enumerating objects: 183, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 183 (delta 105), reused 123 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (183/183), 127.23 KiB | 2.60 MiB/s, done.
Resolving deltas: 100% (105/105), done.
/content/SAM3


## Adim 3 - HuggingFace Login

Sol menude Secrets'a `HF_TOKEN` ekle ve "Notebook access" anahtarini **ac**.

In [5]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("HuggingFace girisi basarili!")

HuggingFace girisi basarili!


## Adim 4 - Veri Setini Yerel Diske Hazirla

Egitim, veriyi **yerel diskten** okur (Drive'dan degil) - binlerce kucuk dosya
okumasi Drive mount'unu oldurdugu icin kritik adim budur.

Mantik:
1. Veri yerel diskte varsa -> atla
2. Drive'da varsa -> yerel diske kopyala (bir kerelik, sirali kopya)
3. Hicbir yerde yoksa -> HuggingFace'ten yerel diske indir + Drive'a yedekle

In [8]:
import subprocess, shutil

def _has_images(base):
    """Verilen klasorde egitim gorselleri var mi?"""
    train_dir = os.path.join(base, "images", "train")
    return os.path.isdir(train_dir) and len(os.listdir(train_dir)) > 0

def _copy_tree(src, dst):
    """Klasoru kopyalar: once rsync (FUSE'a nazik), olmazsa shutil."""
    src = src.rstrip("/")
    dst = dst.rstrip("/")
    os.makedirs(dst, exist_ok=True)
    try:
        subprocess.run(["rsync", "-a", src + "/", dst + "/"], check=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        shutil.copytree(src, dst, dirs_exist_ok=True)

if _has_images(LOCAL_DATA_DIR):
    print("Veri seti zaten yerel diskte (/content/dacl10k) - atlaniyor.")
elif _has_images(DRIVE_DATA_DIR):
    print("Veri seti Drive'dan yerel diske kopyalaniyor (bir kerelik)...")
    _copy_tree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print("Kopyalama bitti. Egitim artik yerel diskten okuyacak (Drive kopmaz).")
else:
    print("Veri seti hicbir yerde yok - HuggingFace'ten yerel diske indiriliyor...")
    !pip install -q datasets
    env = os.environ.copy()
    env["SAM3_DATA_DIR"] = LOCAL_DATA_DIR
    subprocess.run(["python", "scripts/download_dataset.py"],
                   cwd=REPO_DIR, env=env, check=True)
    print("Drive'a yedekleniyor (sonraki oturumlarda tekrar inmesin)...")
    _copy_tree(LOCAL_DATA_DIR, DRIVE_DATA_DIR)
    print("Yedekleme bitti.")

Veri seti Drive'dan yerel diske kopyalaniyor (bir kerelik)...
Kopyalama bitti. Egitim artik yerel diskten okuyacak (Drive kopmaz).


## Adim 5 - Veri Seti Kontrolu

Yerel diskteki veri setinin `images/{train,validation}` klasorleriyle hazir oldugunu kontrol ederiz.

In [9]:
train_img_dir = os.path.join(LOCAL_DATA_DIR, "images", "train")
val_img_dir   = os.path.join(LOCAL_DATA_DIR, "images", "validation")

if not os.path.isdir(train_img_dir):
    raise FileNotFoundError(
        f"Veri seti bulunamadi: {train_img_dir}\n"
        "Adim 4'u calistir veya Drive'da 'MyDrive/sam3/dacl10k/' yapisini kontrol et."
    )

train_count = len([f for f in os.listdir(train_img_dir) if f.lower().endswith(".jpg")])
val_count   = len([f for f in os.listdir(val_img_dir)   if f.lower().endswith(".jpg")])
print(f"Egitim gorseli    : {train_count}")
print(f"Validation gorseli: {val_count}")

Egitim gorseli    : 6935
Validation gorseli: 975


## Adim 6 - Egitimi Arka Planda Baslat (+ Drive Yedekleme + Devam)

Once Drive'daki onceki ilerleme yerel diske geri yuklenir, sonra iki arka plan process baslar:
1. **Egitim** - sadece **yerel diske** yazar -> Drive kopsa bile cokmez.
   **Onceki checkpoint varsa SIFIRDAN baslamaz, en son epoch'tan USTUNE DEVAM eder.**
2. **Yedekleme dongusu** - her 120 sn'de yerel checkpoint+log'u **Drive'a** kopyalar -> ilerleme kalici olur

- `SAM3_DATA_DIR`       -> yerel veri (`/content/dacl10k`)
- `SAM3_CHECKPOINT_DIR` -> yerel checkpoint (`/content/checkpoints`), Drive'a yedeklenir
- `NUM_EPOCHS`          -> toplam kac epoch (devam ederken kalan epoch'lar kosulur)

In [11]:
import subprocess

# Onceki oturumdan kalan checkpoint/log varsa Drive'dan yerel diske geri getir.
# Boylece egitim SIFIRDAN baslamaz, en son epoch'tan USTUNE DEVAM eder.
def _restore(src, dst):
    if os.path.isdir(src) and os.listdir(src):
        os.makedirs(dst, exist_ok=True)
        try:
            subprocess.run(["rsync", "-a", src.rstrip("/") + "/", dst.rstrip("/") + "/"], check=True)
        except (FileNotFoundError, subprocess.CalledProcessError):
            import shutil
            shutil.copytree(src, dst, dirs_exist_ok=True)

print("Onceki ilerleme (varsa) Drive'dan yerel diske geri yukleniyor...")
_restore(DRIVE_CHECKPOINT_DIR, LOCAL_CKPT_DIR)
_restore(DRIVE_LOG_DIR, LOCAL_LOG_DIR)

import json
_state_path = os.path.join(LOCAL_CKPT_DIR, "training_state.json")
if os.path.exists(_state_path):
    _s = json.load(open(_state_path))
    print(f"Devam edilecek: {_s.get('last_epoch', 0)}. epoch'tan (en iyi val_loss: {_s.get('best_val_loss')})")
else:
    print("Onceki ilerleme yok - egitim sifirdan baslayacak.")

env = os.environ.copy()
env["SAM3_DATA_DIR"]       = LOCAL_DATA_DIR    # yerel disk -> Drive kopmaz
env["SAM3_CHECKPOINT_DIR"] = LOCAL_CKPT_DIR    # yerel disk -> yazma her zaman guvenli
env["NUM_EPOCHS"]          = "10"

log_path = os.path.join(LOCAL_LOG_DIR, "faz4_train.log")
log_file = open(log_path, "a")

# 1) Egitim process'i - tamamen yerel diske yazar, kaldigi epoch'tan devam eder
train_proc = subprocess.Popen(
    ["python", "scripts/run_train.py"],
    cwd=REPO_DIR, env=env,
    stdout=log_file, stderr=subprocess.STDOUT,
)
print(f"Egitim arka planda basladi. PID: {train_proc.pid}")

# 2) Yedekleme dongusu - yerel checkpoint+log her 120 sn Drive'a kopyalanir.
#    Drive o an kopuksa hata yutulur (2>/dev/null), bir sonraki turda tekrar dener.
sync_cmd = (
    'while true; do '
    f'rsync -a "{LOCAL_CKPT_DIR}" "{DRIVE_CHECKPOINT_DIR}" 2>/dev/null; '
    f'rsync -a "{LOCAL_LOG_DIR}" "{DRIVE_LOG_DIR}" 2>/dev/null; '
    'sleep 120; done'
)
sync_proc = subprocess.Popen(["bash", "-c", sync_cmd])
print(f"Drive yedekleme dongusu basladi (her 120 sn). PID: {sync_proc.pid}")
print(f"Yerel log: {log_path}")
print("Egitim sadece yerel diske yaziyor -> Drive kopsa bile egitim DURMAZ.")

Onceki ilerleme (varsa) Drive'dan yerel diske geri yukleniyor...
Onceki ilerleme yok - egitim sifirdan baslayacak.
Egitim arka planda basladi. PID: 17982
Drive yedekleme dongusu basladi (her 120 sn). PID: 17983
Yerel log: /content/logs/faz4_train.log
Egitim sadece yerel diske yaziyor -> Drive kopsa bile egitim DURMAZ.


## Adim 7 - Egitimi Takip Et

Bu hucreyi istedigin zaman tekrar calistirarak son loglari gorebilirsin
(egitimi durdurmaz). Ayni oturumda yerel log, yeni oturumda Drive yedegi okunur.

In [24]:
local_log = "/content/logs/faz4_train.log"
drive_log = "/content/drive/MyDrive/sam3/logs/faz4_train.log"
log_path = local_log if os.path.exists(local_log) else drive_log
print("Log dosyasi:", log_path)
!tail -n 40 "{log_path}"

Log dosyasi: /content/logs/faz4_train.log
detector_model.text_encoder.encoder.layers.{0...23}.self_attn.v_proj.bias     | UNEXPECTED | 
detector_model.text_encoder.encoder.layers.{0...23}.self_attn.out_proj.bias   | UNEXPECTED | 
detector_model.text_encoder.encoder.layers.{0...23}.self_attn.q_proj.bias     | UNEXPECTED | 
detector_model.text_encoder.final_layer_norm.bias                             | UNEXPECTED | 
detector_model.text_encoder.encoder.layers.{0...23}.self_attn.q_proj.weight   | UNEXPECTED | 
detector_model.text_encoder.encoder.layers.{0...23}.self_attn.out_proj.weight | UNEXPECTED | 
detector_model.text_encoder.encoder.layers.{0...23}.self_attn.k_proj.weight   | UNEXPECTED | 
detector_model.text_encoder.encoder.layers.{0...23}.mlp.fc1.bias              | UNEXPECTED | 
detector_model.text_encoder.encoder.layers.{0...23}.mlp.fc2.bias              | UNEXPECTED | 
detector_model.text_encoder.encoder.layers.{0...23}.layer_norm1.weight        | UNEXPECTED | 
detector_model.tex

In [23]:
if 'train_proc' in locals() and train_proc.poll() is None:
    print(f"Eğitim süreci (PID: {train_proc.pid}) hala çalışıyor.")
else:
    print("Eğitim süreci çalışmıyor veya tamamlandı.")

if 'sync_proc' in locals() and sync_proc.poll() is None:
    print(f"Drive yedekleme süreci (PID: {sync_proc.pid}) hala çalışıyor.")
else:
    print("Drive yedekleme süreci çalışmıyor veya tamamlandı.")


Eğitim süreci (PID: 17982) hala çalışıyor.
Drive yedekleme süreci (PID: 17983) hala çalışıyor.


## Adim 8 - Checkpointleri Kontrol Et

`epoch_X_lora` ve `best_lora` klasorleri egitim ilerledikce birikir.
Ayni oturumda yerel klasor, yeni oturumda Drive yedegi listelenir.

In [17]:
local_ckpt = "/content/checkpoints/"
drive_ckpt = "/content/drive/MyDrive/sam3/checkpoints/"
ckpt_dir = local_ckpt if (os.path.isdir(local_ckpt) and os.listdir(local_ckpt)) else drive_ckpt
print("Klasor:", ckpt_dir)
for name in sorted(os.listdir(ckpt_dir)):
    print(name)

Klasor: /content/checkpoints/
epoch_1_batch_1000_lora
epoch_1_batch_1500_lora
epoch_1_batch_2000_lora
epoch_1_batch_2500_lora
epoch_1_batch_3000_lora
epoch_1_batch_3500_lora
epoch_1_batch_4000_lora
epoch_1_batch_4500_lora
epoch_1_batch_5000_lora
epoch_1_batch_500_lora
epoch_1_batch_5500_lora
epoch_1_batch_6000_lora


## Ozet

- **Kod + veri yerel diskte** -> Drive mount'u cokmez (`Transport endpoint` hatasi biter)
- **Egitim sadece yerel diske yazar** -> Drive kopsa bile egitim DURMAZ
- **Yedekleme dongusu** her 120 sn checkpoint+log'u Drive'a kopyalar -> ilerleme kalici
- **Kaldigi yerden devam** -> tekrar calistirinca SIFIRDAN baslamaz; `training_state.json`
  ile en son epoch'tan ustune devam eder (Adim 6 once Drive'dan checkpointleri geri yukler)
- Egitim arka plan process -> sekme/baglanti kapanmasi egitimi durdurmaz
- `CHECKPOINT_EVERY_STEPS` (config.py) ile epoch bitmeden de ara checkpoint alinir
- Egitim bittiginde Adim 8 ile checkpointleri, Adim 7 ile loglari kontrol et
- Sirada Faz 5: `best_lora` veya `epoch_X_lora` ile inference (`scripts/run_infer.py`)